In [1]:
import os
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.nvidia import NVIDIAEmbedding
from llama_index.llms.nvidia import NVIDIA
from llama_index.core import Settings
import chromadb
from dotenv import load_dotenv

c:\Users\Pietro\Documents\develop\rag-mtg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# Load environment variables from .env file
load_dotenv()
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

In [3]:
# Reconnect to the existing persistent Chroma database
# This points to the same path where data was saved in the setup phase
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [4]:
# Get the existing collection (use get_collection, not get_or_create_collection)
# This retrieves the collection that was created and populated earlier
# Will raise an error if the collection doesn't exist
chroma_collection = chroma_client.get_collection("documents_collection")

In [5]:
# Wrap the collection in LlamaIndex's vector store adapter
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [6]:
# Create storage context pointing to the existing vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [10]:
# LLM to generate answers based on retrieved information
Settings.llm = NVIDIA(
    model="meta/llama-3.1-8b-instruct",  # o altri modelli disponibili
    api_key=os.getenv("NVIDIA_API_KEY")
)

# Embedding model to convert text into vectors for retrieval
embed_model = NVIDIAEmbedding(
    model="nvidia/nv-embed-v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

In [11]:
# Load the index from the existing vector store
# This reads the previously saved embeddings instead of recomputing them
# Note: from_vector_store() instead of from_documents()
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

In [15]:
# Now you can query the index
# The query engine converts your question to a vector and finds similar document chunks
query_engine = index.as_query_engine()
response = query_engine.query("Can you explain the deathtouch and trample interaction?")
print(response)

When a creature with deathtouch and trample deals damage to a creature, any amount of damage greater than 1 is considered excess damage. According to the trample ability, excess damage is assigned as the controller chooses among the blocking creatures and the player or planeswalker the creature is attacking.

In this scenario, the deathtouch ability doesn't directly affect the assignment of excess damage, but it does influence the definition of excess damage. The excess damage is then assigned according to the trample ability, allowing the controller to choose how to assign the excess damage among the blocking creatures and the player or planeswalker.

So, the interaction between deathtouch and trample is that deathtouch helps determine what constitutes excess damage, while trample dictates how that excess damage is assigned.


In [16]:
# Optional: print retrieved chunks for debugging
print("\n=== CHUNK RETRIEVED ===")
for i, node in enumerate(response.source_nodes):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Score:  {node.score:.4f}")
    print(f"Source: {node.node.metadata}")
    print(f"Testo:  {node.node.get_content()[:300]}...")  # primi 300 caratteri


=== CHUNK RETRIEVED ===

--- Chunk 1 ---
Score:  0.3974
Source: {'file_path': 'c:\\Users\\Pietro\\Documents\\develop\\rag-mtg\\documents\\MagicCompRules_20260417.txt', 'file_name': 'MagicCompRules_20260417.txt', 'file_type': 'text/plain', 'file_size': 971061, 'creation_date': '2026-06-02', 'last_modified_date': '2026-06-02'}
Testo:  Objects can deal damage to battles, creatures, planeswalkers, and players. This is generally detrimental to the object or player that receives that damage. An object that deals damage is the source of that damage.

120.1a Damage can’t be dealt to an object that’s not a battle, a creature, or a pla...

--- Chunk 2 ---
Score:  0.3801
Source: {'file_path': 'c:\\Users\\Pietro\\Documents\\develop\\rag-mtg\\documents\\MagicCompRules_20260417.txt', 'file_name': 'MagicCompRules_20260417.txt', 'file_type': 'text/plain', 'file_size': 971061, 'creation_date': '2026-06-02', 'last_modified_date': '2026-06-02'}
Testo:  702.17. Reach

702.17a Reach is a static ability.

